## Task-adaptive pretraining (TAPT)

### Colab Setup

In [ ]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "ftfy",
            "nltk",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    from google.colab import drive

    drive.mount("/content/drive")

KeyboardInterrupt: 

### Key Imports

In [ ]:
import torch

from config import RESULTS_DIR, SHAH_PLM, SHAH_SEEDS
from data.loader_twd_labelled import load_splits
from models.dapt import dapt
from models.plm_finetune import finetune
from utils.results import already_done, save_result

OUT = RESULTS_DIR / "results.csv"
SEEDS = SHAH_SEEDS
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

### Continued pretraining (one adapted model per seed)

In [ ]:
# TAPT pretrains on the task's own training sentences, labels ignored. 100
# epochs follows Gururangan et al. 2020 section 4.1. one model per seed, since
# each seed has a different train split -- unlike DAPT, which is seed-independent.
ARM = "tapt:roberta-large"
ENC = "roberta-large"
EPOCHS = 100

for seed in SEEDS:
    save_dir = str(RESULTS_DIR / "models" / f"tapt-s{seed}")
    if os.path.isdir(save_dir):
        print(f"{ARM} seed {seed}: already adapted, skipping")
        continue
    train, _ = load_splits("benchmark", seed=seed)
    sentences = train["sentence"].to_list()
    print(f"{ARM} seed {seed}: {len(sentences):,} sentences, {EPOCHS} epochs", flush=True)
    dapt(
        sentences,
        model_name=SHAH_PLM[ENC]["model_name"],
        epochs=EPOCHS,
        save_dir=save_dir,
        device=DEVICE,
        verbose=True,
    )

### Fine-tune adapted encoders (3 seeds)

In [ ]:
cfg = SHAH_PLM[ENC]

for seed in SEEDS:
    if already_done(OUT, force=FORCE, model=ARM, corpus="twd", seed=seed):
        print(f"{ARM} seed {seed}: already done, skipping")
        continue
    train, test = load_splits("benchmark", seed=seed)
    model, tok_, metrics = finetune(
        train,
        model_name=str(RESULTS_DIR / "models" / f"tapt-s{seed}"),
        lr=cfg["lr"],
        batch_size=cfg["batch_size"],
        seed=seed,
        test_df=test,
        device=DEVICE,
        verbose=True,
    )
    save_result(
        OUT,
        model=ARM,
        corpus="twd",
        seed=seed,
        epochs=metrics["epochs"],
        weighted_f1=round(metrics["test_f1"], 4),
        macro_f1=round(metrics["test_macro_f1"], 4),
    )
    print(f"{ARM} seed {seed}: macro={metrics['test_macro_f1']:.4f}")
    del model, tok_
    torch.cuda.empty_cache()